In [1]:
%cd ~/state

# Colab-specific config for pytorch lightning
import os
os.environ['MPLBACKEND'] = 'Agg'
from huggingface_hub import snapshot_download
import scanpy as sc
import numpy as np

/home/averywpx/state


# Downlaod raw data

In [2]:
# Define the repository ID and the local directory to save the files
repo_id = "arcinstitute/Replogle-Nadig-Preprint"
local_dir = "/data/storage/Replogle/raw_data"

# Create the local directory if it doesn't exist
os.makedirs(local_dir, exist_ok=True)

# Download everything except the concatenated file
snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    local_dir=local_dir,
    local_dir_use_symlinks=False,
    ignore_patterns=["replogle.h5ad"],
)

/home/averywpx/jupyter-env/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

'/data/storage/Replogle/raw_data'

Downlaod preprocessed data

In [3]:
from huggingface_hub import snapshot_download
import os

repo_id = "arcinstitute/Replogle-Nadig-Preprint"
local_dir = "/data/storage/Replogle/processed_data"
os.makedirs(local_dir, exist_ok=True)

# Download ONLY the specified file
snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    local_dir=local_dir,
    local_dir_use_symlinks=False,          # optional: copy files instead of symlinks
    allow_patterns=["replogle.h5ad"],  # restrict to just this file
)

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

'/data/storage/Replogle/processed_data'

In [12]:
preprocessed_sub_ds_ntc_adata.obs["gene"].value_counts()

gene
TFAM             1166
non-targeting     500
RHOQ              145
CDK1              134
MYBL2             133
BAP1               98
STAT5B             91
SETD1A             78
MTOR               66
MAP2K7             53
LRP5               50
MYC                49
S100A1             46
BRCA1              44
Name: count, dtype: int64

In [18]:
preprocessed_sub_ds_ntc_adata

AnnData object with n_obs × n_vars = 2653 × 6642
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'cell_line', 'batch'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    obsm: 'X_hvg'

In [16]:
"BRCA1" in genes_list

True

# Read Data

In [37]:
genes_list=["MYC", "MYBL2", "CDK1", "TFAM", "MTOR",
            "STAT5B", "BAP1", "BRCA1", "SETD1A", "NELFA",
           "HAMP", "S100A1", "LRP5", "MAP2K7", "RHOQ"]
control=["non-targeting"]

In [4]:
# input data
hepg2_raw_adata_path = "/data/storage/Replogle/raw_data/GSE264667_hepg2_raw_singlecell_01.h5ad"
preprocessed_adata_path = "/data/storage/Replogle/processed_data/replogle.h5ad"

# intermediate data
preprocessed_hepg2_path = "/data/storage/Replogle/processed_data/replogle_hepg2.h5ad"
processed_sub_hepg2_path = "/data/storage/Replogle/processed_data/replogle_hepg2_sub.h5ad"

# output data
raw_sub_hepg2_path = "/data/storage/Replogle/raw_data/GSE264667_hepg2_raw_singlecell_01_sub.h5ad"
preprocessed_hepg2_sub_ds_ntc_path = "/data/storage/Replogle/processed_data/replogle_hepg2_sub.h5ad"

In [50]:
## Read adata
# hepg2_raw_adata = sc.read_h5ad(hepg2_raw_adata_path)
raw_sub_ds_ntc_adata = sc.read_h5ad(raw_sub_hepg2_path)

# preprocessed_adata = sc.read_h5ad(preprocessed_adata_path)
# preprocessed_hepg2_adata = sc.read_h5ad(preprocessed_hepg2_path)
# preprocessed_sub_ds_ntc_adata = sc.read_h5ad(preprocessed_hepg2_sub_ds_ntc_path)


# Extract raw and preprocessed data for certain gene target

In [39]:
def extract_adata(adta, genes_list, write=False, output_path="./adata.h5ad"):
    # extract certain gene target
    sub_adata = adta[
        adta.obs["gene"].isin(genes_list)
    ].copy()
    
    # separate ntc cells
    ntc_adata = adta[
        adta.obs["gene"]=="non-targeting"
    ].copy()
    
    # downsample to 500
    ds_ntc_data = ntc_adata[
        np.random.choice(ntc_adata.n_obs, min(500, ntc_adata.n_obs), replace=False)
    ].copy()
    
    # concatenate two adata
    sub_ds_ntc_adata = sub_adata.concatenate(ds_ntc_data)

    if write == True:
        sub_ds_ntc_adata.write_h5ad(output_path)

    return ntc_adata, ds_ntc_data, sub_ds_ntc_adata

In [43]:
# check if gene exist in adata
missing = set(genes_list) - set(preprocessed_hepg2_adata.obs["gene"].unique())
print(missing)

set()


In [44]:
raw_ntc_adata, raw_ds_ntc_data, raw_sub_ds_ntc_adata = extract_adata(hepg2_raw_adata, genes_list)
raw_sub_ds_ntc_adata.obs

NameError: name 'hepg2_raw_adata' is not defined

In [40]:
preprocessed_ntc_adata, preprocessed_ds_ntc_data, preprocessed_sub_ds_ntc_adata = extract_adata(preprocessed_hepg2_adata, genes_list)
preprocessed_sub_ds_ntc_adata.obs

/tmp/ipykernel_1833821/4009132929.py:18: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  sub_ds_ntc_adata = sub_adata.concatenate(ds_ntc_data)


,gem_group,gene,gene_id,transcript,gene_transcript,sgID_AB,mitopercent,UMI_count,z_gemgroup_UMI,cell_line,batch
cell_barcode,,,,,,,,,,,
AAACCCAAGATGGCAC-43-0,43,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.071596,23912.0,2.079665,hepg2,0
AAACCCAAGCACACAG-14-0,14,BAP1,ENSG00000163930,P1P2,760_BAP1_P1P2_ENSG00000163930,BAP1_+_52443980.23-P1P2|BAP1_+_52443883.23-P1P2,0.150652,33209.0,0.083693,hepg2,0
AAACCCAAGCGACTGA-47-0,47,MAP2K7,ENSG00000076984,P1P2,4805_MAP2K7_P1P2_ENSG00000076984,MAP2K7_-_7968835.23-P1P2|MAP2K7_+_7968839.23-P1P2,0.097892,15895.0,-0.089397,hepg2,0
AAACCCAAGGGTGAGG-48-0,48,HAMP,ENSG00000105697,P1P2,3626_HAMP_P1P2_ENSG00000105697,HAMP_+_35771624.23-P1P2|HAMP_+_35773493.23-P1P2,0.064252,5245.0,-1.481256,hepg2,0
AAACCCAAGGTCGTCC-18-0,18,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.078540,8989.0,-0.152877,hepg2,0
...,...,...,...,...,...,...,...,...,...,...,...
GGGCTACCAAGTGGCA-5-1,5,non-targeting,non-targeting,non-targeting,11163_non-targeting_non-targeting_non-targeting,non-targeting_02659|non-targeting_00060,0.102279,13512.0,-0.387900,hepg2,1
TCTACCGCAATATCCG-15-1,15,non-targeting,non-targeting,non-targeting,10859_non-targeting_non-targeting_non-targeting,non-targeting_00652|non-targeting_02540,0.092832,21749.0,0.565899,hepg2,1
GTGGGAAGTCAGGTGA-19-1,19,non-targeting,non-targeting,non-targeting,11164_non-targeting_non-targeting_non-targeting,non-targeting_02668|non-targeting_03017,0.094878,19446.0,0.506805,hepg2,1


# Extract preprocessed data for certain gene target

In [ ]:
# preprocessed_hepg2_adata = preprocessed_adata[preprocessed_adata.obs["cell_line"] == "hepg2"].copy()
# preprocessed_hepg2_adata.write_h5ad("/data/storage/Replogle/processed_data/replogle_hepg2.h5ad")

In [93]:
# For preprocessed data
# extract certain gene target
preprocessed_hepg2_sub_adata = preprocessed_hepg2_adata[
    preprocessed_hepg2_adata.obs["gene"].isin(genes_list)
].copy()

# separate ntc cells
preprocessed_ntc_adata = preprocessed_hepg2_adata[
    preprocessed_hepg2_adata.obs["gene"]=="non-targeting"
].copy()

# downsample to 500
preprocessed_ds_ntc = preprocessed_ntc_adata[
    np.random.choice(preprocessed_ds_ntc_adata.n_obs, min(500, preprocessed_ds_ntc_adata.n_obs), replace=False)
].copy()

# concatenate two adata
preprocessed_sub_ds_ntc_adata = preprocessed_hepg2_sub_adata.concatenate(preprocessed_ds_ntc)
preprocessed_sub_ds_ntc_adata.obs

,gem_group,gene,gene_id,transcript,gene_transcript,sgID_AB,mitopercent,UMI_count,z_gemgroup_UMI,cell_line
cell_barcode,,,,,,,,,,
AAACCCAAGATGGCAC-43,43,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.071596,23912.0,2.079665,hepg2
AAACCCAAGCACACAG-14,14,BAP1,ENSG00000163930,P1P2,760_BAP1_P1P2_ENSG00000163930,BAP1_+_52443980.23-P1P2|BAP1_+_52443883.23-P1P2,0.150652,33209.0,0.083693,hepg2
AAACCCAAGCGACTGA-47,47,MAP2K7,ENSG00000076984,P1P2,4805_MAP2K7_P1P2_ENSG00000076984,MAP2K7_-_7968835.23-P1P2|MAP2K7_+_7968839.23-P1P2,0.097892,15895.0,-0.089397,hepg2
AAACCCAAGGTCGTCC-18,18,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.078540,8989.0,-0.152877,hepg2
AAACCCACAAAGGGTC-52,52,STAT5B,ENSG00000173757,P1P2,8499_STAT5B_P1P2_ENSG00000173757,STAT5B_+_40428264.23-P1P2|STAT5B_+_40428385.23...,0.065243,34839.0,3.090645,hepg2
...,...,...,...,...,...,...,...,...,...,...
TTTGTTGAGTCTAGCT-7,7,CDK1,ENSG00000170312,P1P2,1429_CDK1_P1P2_ENSG00000170312,CDK1_-_62538284.23-P1P2|CDK1_+_62538309.23-P1P2,0.089202,22522.0,2.581406,hepg2
TTTGTTGCAGTTGAAA-33,33,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.058227,15096.0,0.090861,hepg2
TTTGTTGGTCTCAGGC-2,2,MYBL2,ENSG00000101057,P1P2,5367_MYBL2_P1P2_ENSG00000101057,MYBL2_+_42295803.23-P1P2|MYBL2_-_42295783.23-P1P2,0.105101,13996.0,-0.284115,hepg2


In [ ]:
import numpy as np

n_cells = 500

idx = np.random.choice(
    adata.n_obs,
    size=min(n_cells, adata.n_obs),
    replace=False
)

adata_sub = adata[
    np.random.choice(
        adata.n_obs,
        size=min(n_cells, adata.n_obs),
        replace=False
)
].copy()

In [71]:
preprocessed_sub_ds_ntc_adata.write_h5ad("/data/storage/Replogle/processed_data/replogle_hepg2_sub.h5ad")

In [16]:
len(preprocessed_hepg2_sub_adata[
    preprocessed_hepg2_sub_adata.obs["gene"].isin(genes_list)
])

1882

In [10]:
preprocessed_adata.X.shape

(643413, 6642)

In [11]:
preprocessed_adata.var

,highly_variable,means,dispersions,dispersions_norm
gene_name_index,,,,
NOC2L,False,0.690140,0.089601,-0.625512
HES4,True,0.693763,1.156615,3.630171
ISG15,True,0.756257,1.104892,3.423880
SDF4,False,0.706835,0.167762,-0.313772
B3GALT6,False,0.283483,0.041215,-0.237526
...,...,...,...,...
MT-ND4L,True,1.379477,0.915507,0.915423
MT-ND4,True,4.453991,2.885937,0.192740
MT-ND5,True,2.873720,1.403472,0.412011


In [13]:
preprocessed_adata

AnnData object with n_obs × n_vars = 643413 × 6642
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'cell_line'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg'
    obsm: 'X_hvg'

In [14]:
preprocessed_adata.obs["gene"].value_counts().head(20)

gene
non-targeting    39165
TFAM              7145
SLC1A5            4284
GFM1              3969
MRPL36            3254
PPP6C             3210
MRPL35            2881
TARDBP            2523
CCDC6             2230
GTF3C4            2049
HSD17B10          1685
FBXO42            1628
GTF2E2            1495
ZDHHC7            1428
PSMB5             1414
TRNT1             1388
TMEM214           1371
TIMM23B           1327
FAM136A           1308
CPSF3             1274
Name: count, dtype: int64

# Extract raw data for certain gene target

In [12]:
hepg2_raw_adata

AnnData object with n_obs × n_vars = 145473 × 9624
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'cell_line'
    var: 'gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'

In [3]:
hepg2_raw_adata.obs

,gem_group,gene,gene_id,transcript,gene_transcript,sgID_AB,mitopercent,UMI_count,z_gemgroup_UMI,cell_line
cell_barcode,,,,,,,,,,
AAACCCAAGAATAGTC-3,3,KIAA1143,ENSG00000163807,P1P2,4360_KIAA1143_P1P2_ENSG00000163807,KIAA1143_+_44803075.23-P1P2|KIAA1143_+_4480308...,0.114029,11234.0,-0.611091,hepg2
AAACCCAAGACAGCTG-12,12,FEN1,ENSG00000168496,P1P2,3057_FEN1_P1P2_ENSG00000168496,FEN1_-_61560380.23-P1P2|FEN1_+_61560617.23-P1P2,0.095229,11068.0,-0.070171,hepg2
AAACCCAAGACCCTTA-53,53,RNPS1,ENSG00000205937,P1P2,7407_RNPS1_P1P2_ENSG00000205937,RNPS1_+_2318108.23-P1P2|RNPS1_+_2318045.23-P1P2,0.086603,16743.0,0.208552,hepg2
AAACCCAAGACGCCCT-39,39,PHF10,ENSG00000130024,P1P2,6279_PHF10_P1P2_ENSG00000130024,PHF10_-_170124315.23-P1P2|PHF10_+_170124573.23...,0.084000,21488.0,0.091101,hepg2
AAACCCAAGAGGCCAT-24,24,HSF1,ENSG00000185122,P1P2,3959_HSF1_P1P2_ENSG00000185122,HSF1_+_145515304.23-P1P2|HSF1_-_145515300.23-P1P2,0.099000,19293.0,0.041390,hepg2
...,...,...,...,...,...,...,...,...,...,...
TTTGTTGTCTTACGGA-22,22,ADAT3,ENSG00000213638,P1,141_ADAT3_P1_ENSG00000213638,ADAT3_-_1905438.23-P1|ADAT3_+_1905424.23-P1,0.093179,16259.0,-0.275738,hepg2
TTTGTTGTCTTCGATT-35,35,DHX16,ENSG00000204560,P1P2,2204_DHX16_P1P2_ENSG00000204560,DHX16_+_30640731.23-P1P2|DHX16_-_30640796.23-P1P2,0.026757,35692.0,-0.256314,hepg2
TTTGTTGTCTTGATTC-31,31,EBNA1BP2,ENSG00000117395,P1P2,2456_EBNA1BP2_P1P2_ENSG00000117395,EBNA1BP2_-_43637779.23-P1P2|EBNA1BP2_+_4363789...,0.066386,25457.0,0.024672,hepg2


In [9]:
hepg2_raw_adata.shape

(145473, 9624)

In [72]:
# For raw data
# extract certain gene target
raw_hepg2_sub_adata = hepg2_raw_adata[
    hepg2_raw_adata.obs["gene"].isin(genes_list)
].copy()

# separate ntc cells
raw_ntc_adata = hepg2_raw_adata[
    hepg2_raw_adata.obs["gene"].isin(control)
].copy()

# downsample to 500
raw_ds_ntc = raw_ntc_adata[
    np.random.choice(raw_ntc_adata.n_obs, min(500, raw_ntc_adata.n_obs), replace=False)
].copy()

# concatenate two adata
raw_sub_ds_ntc_adata = raw_hepg2_sub_adata.concatenate(raw_ds_ntc)
raw_sub_ds_ntc_adata.obs

,gem_group,gene,gene_id,transcript,gene_transcript,sgID_AB,mitopercent,UMI_count,z_gemgroup_UMI,cell_line
cell_barcode,,,,,,,,,,
AAACCCAAGATGGCAC-43,43,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.071596,23912.0,2.079665,hepg2
AAACCCAAGCGACTGA-47,47,MAP2K7,ENSG00000076984,P1P2,4805_MAP2K7_P1P2_ENSG00000076984,MAP2K7_-_7968835.23-P1P2|MAP2K7_+_7968839.23-P1P2,0.097892,15895.0,-0.089397,hepg2
AAACCCAAGGGTGAGG-48,48,HAMP,ENSG00000105697,P1P2,3626_HAMP_P1P2_ENSG00000105697,HAMP_+_35771624.23-P1P2|HAMP_+_35773493.23-P1P2,0.064252,5245.0,-1.481256,hepg2
AAACCCAAGGTCGTCC-18,18,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.078540,8989.0,-0.152877,hepg2
AAACCCACAATAGTCC-31,31,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.053891,21395.0,-0.311764,hepg2
...,...,...,...,...,...,...,...,...,...,...
TTTGTTGAGAGTCAAT-11,11,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.049587,11132.0,-0.276612,hepg2
TTTGTTGAGTCTAGCT-7,7,CDK1,ENSG00000170312,P1P2,1429_CDK1_P1P2_ENSG00000170312,CDK1_-_62538284.23-P1P2|CDK1_+_62538309.23-P1P2,0.089202,22522.0,2.581406,hepg2
TTTGTTGCAGTTGAAA-33,33,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.058227,15096.0,0.090861,hepg2


In [ ]:
raw_sub_ds_ntc_adata.write_h5ad("/data/storage/Replogle/raw_data/GSE264667_hepg2_raw_singlecell_01_sub.h5ad")

/tmp/ipykernel_770011/2917828112.py:11: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  raw_sub_ds_ntc_adata = raw_hepg2_sub_adata.concatenate(raw_ds_ntc)


AnnData object with n_obs × n_vars = 2746 × 9624
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'cell_line', 'batch'
    var: 'gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'

In [6]:
raw_hepg2_sub_adata

AnnData object with n_obs × n_vars = 6946 × 9624
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'cell_line'
    var: 'gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'

# Calculate KD percentage

In [26]:
raw_sub_ds_ntc_adata.obs

,gem_group,gene,gene_id,transcript,gene_transcript,sgID_AB,mitopercent,UMI_count,z_gemgroup_UMI,cell_line,batch
cell_barcode,,,,,,,,,,,
AAACCCAAGATGGCAC-43-0,43,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.071596,23912.0,2.079665,hepg2,0
AAACCCAAGCGACTGA-47-0,47,MAP2K7,ENSG00000076984,P1P2,4805_MAP2K7_P1P2_ENSG00000076984,MAP2K7_-_7968835.23-P1P2|MAP2K7_+_7968839.23-P1P2,0.097892,15895.0,-0.089397,hepg2,0
AAACCCAAGGGTGAGG-48-0,48,HAMP,ENSG00000105697,P1P2,3626_HAMP_P1P2_ENSG00000105697,HAMP_+_35771624.23-P1P2|HAMP_+_35773493.23-P1P2,0.064252,5245.0,-1.481256,hepg2,0
AAACCCAAGGTCGTCC-18-0,18,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.078540,8989.0,-0.152877,hepg2,0
AAACCCACAATAGTCC-31-0,31,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.053891,21395.0,-0.311764,hepg2,0
...,...,...,...,...,...,...,...,...,...,...,...
CTACATTCATAATCCG-17-1,17,non-targeting,non-targeting,non-targeting,10967_non-targeting_non-targeting_non-targeting,non-targeting_01376|non-targeting_00354,0.077831,23371.0,-0.227791,hepg2,1
GAGACCCTCGCCAACG-35-1,35,non-targeting,non-targeting,non-targeting,11195_non-targeting_non-targeting_non-targeting,non-targeting_02855|non-targeting_01051,0.095688,50717.0,0.573676,hepg2,1
TCTCTGGCACAGTATC-52-1,52,non-targeting,non-targeting,non-targeting,11232_non-targeting_non-targeting_non-targeting,non-targeting_03135|non-targeting_02838,0.102283,30787.0,2.467425,hepg2,1


In [23]:
preprocessed_sub_ds_ntc_adata

AnnData object with n_obs × n_vars = 2382 × 6642
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'cell_line', 'batch'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    obsm: 'X_hvg'

In [34]:
gene="MYC"
idx = raw_sub_ds_ntc_adata.var_names.get_loc("ENSG00000188976")
idx
# raw_sub_ds_ntc_adata.X[:, "ENSG00000228794"]

1

In [52]:
preprocessed_sub_ds_ntc_adata.var_names

Index(['NOC2L', 'HES4', 'ISG15', 'SDF4', 'B3GALT6', 'UBE2J2', 'ACAP3', 'PUSL1',
       'INTS11', 'DVL1',
       ...
       'MT-CO2', 'MT-ATP8', 'MT-ATP6', 'MT-CO3', 'MT-ND3', 'MT-ND4L', 'MT-ND4',
       'MT-ND5', 'MT-ND6', 'MT-CYB'],
      dtype='object', name='gene_name_index', length=6642)

In [ ]:
gene="MYC"
idx = preprocessed_sub_ds_ntc_adata.var_names.get_loc("ENSG00000188976")
idx

In [44]:
raw_token_map = dict(raw_sub_ds_ntc_adata.obs[["gene", "gene_id"]].drop_duplicates().values)
raw_token_map

{'TFAM': 'ENSG00000108064',
 'MAP2K7': 'ENSG00000076984',
 'HAMP': 'ENSG00000105697',
 'S100A1': 'ENSG00000160678',
 'LRP5': 'ENSG00000162337',
 'MYBL2': 'ENSG00000101057',
 'MTOR': 'ENSG00000198793',
 'CDK1': 'ENSG00000170312',
 'MYC': 'ENSG00000136997',
 'RHOQ': 'ENSG00000119729',
 'non-targeting': 'non-targeting'}

In [47]:
processed_token_map = dict(preprocessed_sub_ds_ntc_adata.obs[["gene", "gene_id"]].drop_duplicates().values)
processed_token_map

{'TFAM': 'ENSG00000108064',
 'MAP2K7': 'ENSG00000076984',
 'HAMP': 'ENSG00000105697',
 'S100A1': 'ENSG00000160678',
 'LRP5': 'ENSG00000162337',
 'MYBL2': 'ENSG00000101057',
 'MTOR': 'ENSG00000198793',
 'CDK1': 'ENSG00000170312',
 'MYC': 'ENSG00000136997',
 'RHOQ': 'ENSG00000119729',
 'non-targeting': 'non-targeting'}

In [55]:
def knockdown(target_gene, adata, data, obs_col="gene_target", ntc="Non-Targeting"):
    if data =="raw":
        ensg_id = raw_token_map[target_gene]
        idx = adata.var_names.get_loc(ensg_id)
    else:
        idx = adata.var_names.get_loc(target_gene)

    
    gene_adata = adata[adata.obs[obs_col]==target_gene].copy()
    ntc_adata = adata[adata.obs[obs_col]==ntc].copy()

    gene_expression = gene_adata.X[:, idx]
    ntc_gene_expression = ntc_adata.X[:, idx]

    

    # pert = adata.obs[obs_col] == target_gene
    # ctrl = adata.obs[obs_col] == ntc
    # print(idx)
    # print(pert)

    pert_expr = gene_expression.mean()
    ctrl_expr = ntc_gene_expression.mean()

    log2FC = np.log2((pert_expr + 1e-6) / (ctrl_expr + 1e-6))
    knockdown = 1 - pert_expr / ctrl_expr
    
    print(target_gene)
    print(f"Mean target expression in perturbed: {pert_expr}")
    print(f"Mean target expression in control: {ctrl_expr}")
    print(f"log2FC: {log2FC}")
    print(f"knockdown: {knockdown}")
    print("\n")
    

    return {
        "gene": target_gene,
        "log2FC": log2FC,
        "knockdown": knockdown
    }

In [78]:
results = []
for gene in genes_list[:-1]:
    result_dict = knockdown(gene, raw_sub_ds_ntc_adata, "raw", obs_col="gene", ntc="non-targeting")
    results.append(result_dict)

results

MYC
Mean target expression in perturbed: 0.3076923191547394
Mean target expression in control: 2.052000045776367
log2FC: -2.737466335296631
knockdown: 0.8500524759292603


MYBL2
Mean target expression in perturbed: 0.11764705926179886
Mean target expression in control: 2.1740000247955322
log2FC: -4.207803249359131
knockdown: 0.9458845257759094


CDK1
Mean target expression in perturbed: 0.2733812928199768
Mean target expression in control: 2.4539999961853027
log2FC: -3.1661441326141357
knockdown: 0.8885976672172546


TFAM
Mean target expression in perturbed: 0.21269579231739044
Mean target expression in control: 2.308000087738037
log2FC: -3.4397735595703125
knockdown: 0.9078441262245178


MTOR
Mean target expression in perturbed: 0.23456789553165436
Mean target expression in control: 0.9020000100135803
log2FC: -1.943117380142212
knockdown: 0.7399469017982483




KeyError: 'PRKCA'

In [50]:
genes_list[1:-1]

['MYBL2', 'CDK1', 'TFAM', 'MTOR', 'HAMP', 'S100A1', 'LRP5', 'MAP2K7']

In [76]:
results = []
for gene in genes_list[:-1]:
    print(gene)
    result_dict = knockdown(gene, preprocessed_sub_ds_ntc_adata, "processed", obs_col="gene", ntc="non-targeting")
    results.append(result_dict)

results

MYC
MYC
Mean target expression in perturbed: 0.09259430319070816
Mean target expression in control: 0.6067663431167603
log2FC: -2.712132453918457
knockdown: 0.8473970890045166


MYBL2
MYBL2
Mean target expression in perturbed: 0.008471238426864147
Mean target expression in control: 0.5672177076339722
log2FC: -6.065018177032471
knockdown: 0.9850652813911438


CDK1
CDK1
Mean target expression in perturbed: 0.01923966035246849
Mean target expression in control: 0.4846641421318054
log2FC: -4.654757976531982
knockdown: 0.9603031277656555


TFAM
TFAM
Mean target expression in perturbed: 0.0402241051197052
Mean target expression in control: 0.6182818412780762
log2FC: -3.942099094390869
knockdown: 0.9349421262741089


MTOR
MTOR
Mean target expression in perturbed: 0.0
Mean target expression in control: 0.30680009722709656
log2FC: -18.226943969726562
knockdown: 1.0


PTEN
PTEN
Mean target expression in perturbed: 0.0
Mean target expression in control: 0.37987345457077026
log2FC: -18.53516387939

KeyError: 'FOXO1'

# Calculate pearson correlation

In [45]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from scipy import sparse

In [59]:
def calc_pearson_correlation(gene_name, adata):
    # WARNING: do log-normalizing for raw counts
    target = gene_name
    target_col = "gene"           
    control_label = "non-targeting"  
    
    # select cells perturbed on given target genes and control cells
    target_cells = adata[adata.obs[target_col] == target].copy()
    nt_cells = adata[adata.obs[target_col] == control_label].copy()
    
    # randomly split target gene cells into two groups
    np.random.seed(1)
    idx = np.random.permutation(target_cells.n_obs)
    half = len(idx) // 2
    
    group1 = target_cells[idx[:half]]
    group2 = target_cells[idx[half:]]
    
    #  calculate mean expression
    def mean_expr(x):
        X = x.X
        if sparse.issparse(X):
            X = X.toarray()
        return np.asarray(X).mean(axis=0)
    
    mean_g1 = mean_expr(group1)
    mean_g2 = mean_expr(group2)
    mean_nt = mean_expr(nt_cells)
    
    # calculate gene expression vectors
    vec_g1 = mean_g1 - mean_nt
    vec_g2 = mean_g2 - mean_nt
    
    # calculate Pearson correlation
    r, p = pearsonr(vec_g1, vec_g2)
    
    print("Pearson r:", r)
    print("p-value:", p)
    return r, p

In [60]:
calc_pearson_correlation("MYC", preprocessed_sub_ds_ntc_adata)

Pearson r: 0.6167507
p-value: 0.0


(np.float32(0.6167507), np.float32(0.0))

In [56]:
result = pd.DataFrame({
    "gene": "MYC",
    "group1_delta": vec_g1,
    "group2_delta": vec_g2
})

result["group1_direction"] = np.where(result["group1_delta"] > 0, "up", "down")
result["group2_direction"] = np.where(result["group2_delta"] > 0, "up", "down")

result.head()

,gene,group1_delta,group2_delta,group1_direction,group2_direction
0,MYC,-0.153586,-0.130436,down,down
1,MYC,-0.053912,0.047878,down,up
2,MYC,0.182204,0.196742,up,up
3,MYC,0.141727,0.196835,up,up
4,MYC,-0.017127,-0.052652,down,down


In [57]:
result

,gene,group1_delta,group2_delta,group1_direction,group2_direction
0,MYC,-0.153586,-0.130436,down,down
1,MYC,-0.053912,0.047878,down,up
2,MYC,0.182204,0.196742,up,up
3,MYC,0.141727,0.196835,up,up
4,MYC,-0.017127,-0.052652,down,down
...,...,...,...,...,...
6637,MYC,-0.071331,0.048681,down,up
6638,MYC,0.080487,-0.065922,up,down
6639,MYC,-0.060722,-0.013212,down,down
6640,MYC,-0.227509,-0.082800,down,down


In [52]:
raw_sub_ds_ntc_adata.X.max()

np.float32(2864.0)

In [51]:
raw_sub_ds_ntc_adata.X[:5, :5]

array([[0., 3., 0., 2., 3.],
       [0., 2., 0., 0., 2.],
       [1., 0., 0., 1., 0.],
       [0., 2., 0., 1., 0.],
       [1., 1., 0., 3., 2.]], dtype=float32)

In [48]:
preprocessed_sub_ds_ntc_adata.layers["counts"]

KeyError: 'counts'